# The masked-convolutional autoregressive model — walkthrough

Interactive walkthrough of the autoregressive masked-convolutional model of
honeybee nest contents from

> Marting PR, Graving JM, et al. *Morphogenesis, differentiation, and robust
> multi-scale organization of the honeybee nest revealed by a novel 4D atlas.*
> **Current Biology** (2026). DOI 10.1016/j.cub.2026.10.064

It reproduces, from the committed weights in `models/`, the numbers behind
**Figure 6** (mean predicted likelihood of the true content class, by path)
and checks them against the committed `figures/` and `results/` directories.

**Setup** (once, from the repository root):

```bash
uv sync --all-extras
```

**Data**: the model's detailed 21-class colony array is committed as
`data/AllColoniesCombinedDS48v2.zip` (zipped because the raw 156 MB `.npy`
exceeds GitHub's 100 MB/file cap). The cell below loads the zip directly, so
no download or unzip step is needed.

**Device**: the notebooks use the first *visible* CUDA GPU by default; select
a physical card with `CUDA_VISIBLE_DEVICES` (e.g. `CUDA_VISIBLE_DEVICES=1`).

In [1]:
import os
import time

import numpy as np
import pandas as pd
import torch

from combinator import (
    build_eval_hives,
    evaluate_spatial_classifier,
    load_grids,
    map_all_labels,
    process_grids,
)
from combinator.labels import COLONY_NAMES, SIMPLE_NAMES
from combinator.models import load_any
from combinator.registry import V2_MODELS
from combinator.results_io import load_results

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

device: cuda:0


## 1 · Load the detailed masks and run the spatial pipeline

`data/AllColoniesCombinedDS48v2.zip` holds one per-colony mask stack
`(weeks, 20, 77, 116)` with the 21 detailed content classes (the raw
`AllColoniesCombinedDS48v2.npy` is inside the zip). The pipeline
remaps to the model's 6 classes, phase-correlation-aligns every frame to a
global mean-wood template, and crops to the mean-wood bounding box --
ending at the **(weeks, 20, 52, 104)** geometry the weights expect.
20 = 10 frames x 2 sides (interleaved) along the depth axis.

In [2]:
# The detailed 21-class colony array.  It is committed zipped (the raw
# 156 MB .npy exceeds GitHub's 100 MB/file cap); load_grids() reads the
# .zip directly.  Point DATA at a raw .npy instead if you have one.
DATA = "data/AllColoniesCombinedDS48v2.zip"

hive_grids = load_grids(DATA)
print("colonies:", len(hive_grids), COLONY_NAMES)
print("weeks per colony:", [g.shape[0] for g in hive_grids])
print("per-week shape:", hive_grids[0][0].shape, hive_grids[0].dtype)

colonies: 12 ['DD1', 'DD2', 'DD4', 'DD5', 'DD6', 'DD7', 'SH2', 'SH3', 'SH4', 'SH5', 'SH6', 'SH7']
weeks per colony: [20, 20, 20, 20, 20, 17, 20, 20, 14, 20, 20, 17]
per-week shape: (20, 77, 116) int32


In [3]:
# 21 detailed classes -> the model's 6 classes
hive_grids = map_all_labels(hive_grids)

# align every frame to the global mean-wood template, then crop each
# colony to its mean-wood bounding box (margin 1, threshold 0.5)
hive_grids = process_grids(hive_grids)

print("processed per-week shape:", hive_grids[0].shape)   # (weeks, 20, 52, 104)
print("classes:", SIMPLE_NAMES)

Zero variance detected in grid 1, frame indices (0, 2)
Zero variance detected in grid 1, frame indices (0, 2)
Zero variance detected in grid 4, frame indices (0, 9)
Zero variance detected in grid 4, frame indices (0, 9)
Zero variance detected in grid 9, frame indices (0, 3)
Zero variance detected in grid 9, frame indices (0, 3)
processed per-week shape: (20, 20, 52, 104)
classes: ['background', 'wood', 'empty_comb', 'any_brood', 'pollen', 'nectar_honey']


## 2 · Build the 18-hive evaluation corpus

* **0–5 control** — `DD1…DD7` (frames in natural order)
* **6–11 treatment** — `SH2…SH7` (frames physically moved within the hive)
* **12–17 simulated** — the controls with their ten frame-*pairs*
  (A+B of each frame) randomly permuted along depth, seeds 42+0…5

Simulating the physical shuffling on the controls gives a direct behavioral
test: if depth position matters, the simulated shuffles should look broken to
a model that uses depth context.

In [4]:
hives = build_eval_hives(hive_grids, num_control=6, seed=42)

pd.DataFrame({
    "hive_id": range(18),
    "path": ["control"] * 6 + ["treatment (physical)"] * 6 + ["simulated"] * 6,
    "source colony": COLONY_NAMES[:6] + COLONY_NAMES[6:] + COLONY_NAMES[:6],
    "weeks": [h.shape[0] for h in hives],
})

## 3 · Evaluate the full 3-D model

`load_any` infers the architecture from the filename (the eight committed
checkpoints are enumerated in `combinator/registry.py`). Evaluation returns,
per hive: ground truth, argmax predictions, per-voxel entropy, and the
negative log-likelihood of the true class (i.e. `-log P(true)`).

In [5]:
key_3d = (3, True, True, True, True, True, 1)
model = load_any("models/spatial_classifier_v2_k3_dTrue_hTrue_wTrue_posTrue_ctxTrue_int1.pth",
                 device=DEVICE)
results = evaluate_spatial_classifier(model, hives, device=DEVICE)

# The paper's Figure 6B uses hive 4, week 10; run the model forward on that
# week to get per-class probabilities (the evaluator keeps only the true-class NLL).
hive_idx, week = 4, 10
with torch.no_grad():
    x = torch.tensor(hives[hive_idx][week:week + 1], dtype=torch.long, device=DEVICE)
    logits = model(x)                                  # (1, 6, 20, 52, 104)
    probs = torch.softmax(logits, dim=1)[0].permute(1, 2, 3, 0).cpu().numpy()  # (20,52,104,6)

gt = results["hives"][hive_idx]["ground_truth"][week]
pred = results["hives"][hive_idx]["predictions"][week]

rows = []
for cls, name in enumerate(SIMPLE_NAMES):
    m = gt == cls
    if not m.any():
        continue
    rows.append({
        "class": name,
        "true voxels": int(m.sum()),
        "voxel accuracy": round(float((pred[m] == cls).mean()), 3),
        "mean P(class | voxel)": round(float(probs[m, cls].mean()), 3),
    })
print(f"hive {hive_idx} (control), week {week}: overall voxel accuracy "
      f"{(pred == gt).mean():.3f}")
pd.DataFrame(rows).set_index("class")

hive 4 (control), week 10: overall voxel accuracy 0.939


true voxels  voxel accuracy  mean P(class | voxel)
class
background          48307           0.998                  0.997
wood                15777           0.999                  0.998
empty_comb          27607           0.811                  0.786
any_brood           12852           0.910                  0.820
pollen               2134           0.993                  0.958
nectar_honey         1483           0.913                  0.826

## 4 · All eight paper models, one table

Run every committed checkpoint over all 18 hives and average voxel accuracy
per configuration group — the same grouping Figure 6 plots (1-D and 2-D
averaged, 3-D and the k=1 baseline shown directly).

In [6]:
all_results = {}
t0 = time.time()
for fname, key in V2_MODELS:
    m = load_any(os.path.join("models", fname), device=DEVICE)
    all_results[key] = evaluate_spatial_classifier(m, hives, device=DEVICE)
    del m
    if DEVICE.startswith("cuda"):
        torch.cuda.empty_cache()
    print(f"  {str(key):34s} {time.time() - t0:5.0f}s")

groups = {
    "k1 baseline": [(1, False, False, False, False, False, 0)],
    "1D": [(3, True, False, False, True, True, 1),
           (3, False, True, False, True, True, 1),
           (3, False, False, True, True, True, 1)],
    "2D": [(3, True, True, False, True, True, 1),
           (3, True, False, True, True, True, 1),
           (3, False, True, True, True, True, 1)],
    "3D": [(3, True, True, True, True, True, 1)],
}
paths = {"control": slice(0, 6), "treatment": slice(6, 12), "simulated": slice(12, 18)}

rows = []
for gname, keys in groups.items():
    for pname, sl in paths.items():
        accs = []
        for key in keys:
            r = all_results[key]["hives"]
            for i in range(sl.start, sl.stop):
                accs.append(float((r[i]["predictions"]
                                   == r[i]["ground_truth"]).mean()))
        rows.append({"config group": gname, "path": pname,
                     "mean voxel accuracy": round(float(np.mean(accs)), 3)})
pd.DataFrame(rows)

(1, False, False, False, False, False, 0)     1s
(3, False, False, True, True, True, 1)     6s
(3, False, True, False, True, True, 1)     8s
(3, True, False, False, True, True, 1)    12s
(3, True, False, True, True, True, 1)    16s
(3, True, True, False, True, True, 1)    18s
(3, False, True, True, True, True, 1)    21s
(3, True, True, True, True, True, 1)    25s


config group       path  mean voxel accuracy
0   k1 baseline    control                0.367
1   k1 baseline  treatment                0.339
2   k1 baseline  simulated                0.367
3            1D    control                0.861
4            1D  treatment                0.849
5            1D  simulated                0.856
6            2D    control                0.888
7            2D  treatment                0.877
8            2D  simulated                0.887
9            3D    control                0.951
10           3D  treatment                0.867
11           3D  simulated                0.879

## 5 · Reproduce Figure 6's likelihood table

Figure 6's y-axis is the **mean predicted likelihood of the true content
class** — averaged over the voxels of each class, over weeks and hives, and
over the configurations in each dimensionality group. Rebuild the table
`figures/mean_likelihood_figure6.csv` from your own evaluation and compare
with the committed numbers.

In [7]:
class_names = ["Background", "Wood", "Empty Comb", "Brood", "Pollen", "Nectar/Honey"]
dim_groups = {
    "1D": [(3, True, False, False, True, True, 1), (3, False, True, False, True, True, 1),
           (3, False, False, True, True, True, 1)],
    "2D": [(3, True, True, False, True, True, 1), (3, True, False, True, True, True, 1),
           (3, False, True, True, True, True, 1)],
    "3D": [(3, True, True, True, True, True, 1)],
}


def mean_likelihood(hives, cls):
    # Mean, over weeks and voxels of class `cls`, of P(true class).
    vals = []
    for hive in hives:
        gt = hive["ground_truth"]
        probs = np.exp(hive["log_likelihood"])  # P(true) per voxel
        for t in range(gt.shape[0]):
            m = gt[t].flatten() == cls
            if np.any(m):
                vals.append(probs[t].reshape(-1)[m].mean())
    return float(np.mean(vals)) if vals else float("nan")


key_k1 = (1, False, False, False, False, False, 0)
rows = []
for dim, keys in dim_groups.items():
    for cls, cname in enumerate(class_names):
        row = {"dimensionality": dim, "class": cname}
        for pname, sl in paths.items():
            per_cfg = [mean_likelihood(all_results[k]["hives"][sl], cls) for k in keys]
            per_cfg = [v for v in per_cfg if not np.isnan(v)]
            row[pname] = round(float(np.mean(per_cfg)), 4)
            row[f"baseline_{pname}"] = round(
                mean_likelihood(all_results[key_k1]["hives"][sl], cls), 4)
        rows.append(row)
fresh = pd.DataFrame(rows)
committed = pd.read_csv("figures/mean_likelihood_figure6.csv")
merged = fresh.merge(committed, on=["dimensionality", "class"],
                     suffixes=("", "_committed"))

cols = [c for c in fresh.columns if c not in ("dimensionality", "class")]
max_diff = float(np.max([abs(merged[c].values - merged[c + "_committed"].values).max()
                         for c in cols]))
print(f"Figure 6 table reproduced; max |fresh - committed| = {max_diff:.2e}")
merged

Figure 6 table reproduced; max |fresh - committed| = 0.00e+00


dimensionality  ...  baseline_simulated_committed
0              1D  ...                        0.1686
1              1D  ...                        0.1678
2              1D  ...                        0.1680
3              1D  ...                        0.1664
4              1D  ...                        0.1644
5              1D  ...                        0.1648
6              2D  ...                        0.1686
7              2D  ...                        0.1678
8              2D  ...                        0.1680
9              2D  ...                        0.1664
10             2D  ...                        0.1644
11             2D  ...                        0.1648
12             3D  ...                        0.1686
13             3D  ...                        0.1678
14             3D  ...                        0.1680
15             3D  ...                        0.1664
16             3D  ...                        0.1644
17             3D  ...                        0.1

## 6 · Cross-check the committed `results/` directory

`results/` holds the full per-hive, per-voxel outputs of all eight models in
144 small npz files plus SHA-256 checksums. `load_results` reassembles the
original dict (verifying every hash) — compare the 3-D model's hive 4 against
the fresh evaluation from step 3.

In [8]:
results_committed = load_results("results")   # verifies every SHA-256
print("configs:", len(results_committed),
      "| hives:", sum(len(v['hives']) for v in results_committed.values()))

a = results_committed[key_3d]["hives"][hive_idx]
b = all_results[key_3d]["hives"][hive_idx]
for field in ("ground_truth", "predictions", "entropy", "log_likelihood"):
    d = np.abs(np.asarray(a[field], float) - np.asarray(b[field], float)).max()
    print(f"  {field:15s} max|delta| = {d:.3e}")

configs: 8 | hives: 144
ground_truth    max|delta| = 0.000e+00
predictions     max|delta| = 0.000e+00
entropy         max|delta| = 4.882e-04
log_likelihood  max|delta| = 7.769e-03


## What next

* `uv run python -m combinator.make_figures --data $DATA --models-dir models
  --outdir figures_out` regenerates the full figure set (PDF/PNG/CSV),
  including the Figure 6 likelihood plot with 95% CIs
  (`n_bootstrap=1000`).
* `uv run python -m combinator.results_io verify results` checks the committed
  results directory byte-for-byte.
* `CombContent_Make.ipynb` / `CombContent_Analysis.ipynb` walk through the
  comb-content (nearest-neighbor dispersion) analyses on the deposited
  `CombMasks_*.npy` and `df_*.csv` files.
* `models/MANIFEST.md` documents the provenance of every checkpoint.